# 03: Overfitting, Underfitting, and Generalization

This notebook visualizes the core problems in ML:
- **Underfitting**: Model is too simple to capture the pattern
- **Overfitting**: Model memorizes training data instead of learning the pattern
- **Good generalization**: Model performs well on unseen data

We'll use polynomial regression to demonstrate these concepts visually.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.metrics import mean_squared_error

## Part 1: Polynomial Regression Demo

We'll generate data from a true quadratic function and try to fit it with polynomials of different degrees.

In [ ]:
# Generate data from a true quadratic function with noise
np.random.seed(42)

def true_function(x):
    return 0.5 * x**2 - 3 * x + 2 + np.random.normal(0, 0.5, x.shape)

n_samples = 30
X = np.sort(np.random.uniform(-2, 4, n_samples)).reshape(-1, 1)
y = true_function(X.ravel())

# Also generate a smooth curve for the true function
X_true = np.linspace(-2, 4, 200).reshape(-1, 1)
y_true = 0.5 * X_true.ravel()**2 - 3 * X_true.ravel() + 2

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

In [ ]:
# Fit polynomials of different degrees
degrees = [1, 3, 10]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, degree in enumerate(degrees):
    ax = axes[i]
    
    # Create polynomial features and fit
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_pred_smooth = model.predict(X_true)
    
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    
    # Plot
    ax.scatter(X_train, y_train, color="#3498db", s=40, label="Train", zorder=5)
    ax.scatter(X_test, y_test, color="#e74c3c", s=40, label="Test", zorder=5)
    ax.plot(X_true, y_true, "g--", linewidth=2, label="True function", alpha=0.7)
    ax.plot(X_true, y_pred_smooth, color="#8e44ad", linewidth=2, label=f"Degree {degree}")
    
    status = "Underfitting" if degree == 1 else ("Overfitting" if degree == 10 else "Good fit")
    color = "#e74c3c" if status != "Good fit" else "#27ae60"
    ax.set_title(f"Degree {degree} — {status}\n"
                 f"Train MSE: {train_mse:.3f} | Test MSE: {test_mse:.3f}",
                 color=color, fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(fontsize=8)
    ax.set_ylim(-4, 12)

plt.suptitle("Polynomial Regression: Underfitting vs Overfitting", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 2: Learning Curves

Learning curves show how model performance changes as we add more training data. They are the best diagnostic tool for overfitting and underfitting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, degree in enumerate([1, 3, 10]):
    ax = axes[i]
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring="neg_mean_squared_error"
    )
    
    train_mean = -train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std,
                    train_mean + train_std, alpha=0.1, color="#3498db")
    ax.fill_between(train_sizes, val_mean - val_std,
                    val_mean + val_std, alpha=0.1, color="#e74c3c")
    ax.plot(train_sizes, train_mean, "o-", color="#3498db", label="Train")
    ax.plot(train_sizes, val_mean, "o-", color="#e74c3c", label="Validation")
    
    status = "Underfitting" if degree == 1 else ("Overfitting" if degree == 10 else "Good fit")
    ax.set_title(f"Degree {degree} — {status}", fontsize=11)
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("MSE")
    ax.legend(loc="upper right")
    ax.set_ylim(bottom=0)

plt.suptitle("Learning Curves: Diagnosing Bias vs Variance", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### How to read learning curves:

**Underfitting (high bias):**
- Both train and validation error are high
- Error doesn't improve much with more data
- Fix: use a more complex model

**Overfitting (high variance):**
- Train error is low, validation error is much higher
- Gap between curves is large
- May improve with more data
- Fix: more data, regularization, simpler model

**Good fit:**
- Both errors converge to a low value
- Small gap between curves

## Part 3: Cross-Validation

Cross-validation gives a more robust estimate of model performance than a single train/test split. It splits the data into k folds and trains/tests k times.

In [ ]:
# Compare models using cross-validation
degrees = range(1, 15)
cv_scores_mean = []
cv_scores_std = []
train_scores_mean = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    
    # Cross-validation scores (negative MSE)
    scores = cross_val_score(model, X_train, y_train,
                             cv=5, scoring="neg_mean_squared_error")
    cv_scores_mean.append(-scores.mean())
    cv_scores_std.append(scores.std())
    
    # Training score (for comparison)
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    train_scores_mean.append(mean_squared_error(y_train, train_pred))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(list(degrees), cv_scores_mean, yerr=cv_scores_std,
            fmt="o-", capsize=4, label="Cross-Val MSE", color="#e74c3c")
ax.plot(list(degrees), train_scores_mean, "s-", label="Train MSE", color="#3498db")
ax.set_xlabel("Polynomial Degree")
ax.set_ylabel("Mean Squared Error")
ax.set_title("Cross-Validation: Finding Optimal Model Complexity")
ax.legend()
ax.set_yscale("log")
ax.set_xticks(list(degrees))

# Highlight the best degree
best_degree = list(degrees)[np.argmin(cv_scores_mean)]
ax.axvline(x=best_degree, color="green", linestyle="--", alpha=0.7,
           label=f"Best degree: {best_degree}")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Best polynomial degree by CV: {best_degree}")
print(f"Best CV MSE: {min(cv_scores_mean):.4f}")

## Part 4: Regularization

Regularization adds a penalty for complex models. It helps prevent overfitting by discouraging large coefficients.

- **Ridge (L2)**: Penalty = sum of squared coefficients. Shrinks coefficients toward zero.
- **Lasso (L1)**: Penalty = sum of absolute coefficients. Can set some coefficients exactly to zero (feature selection).

The regularization strength is controlled by the parameter `alpha`.

In [ ]:
# Overfitting with degree 10 polynomial
overfit_model = make_pipeline(
    PolynomialFeatures(10),
    LinearRegression()
)
overfit_model.fit(X_train, y_train)

# Same model with Ridge regularization
ridge_model = make_pipeline(
    PolynomialFeatures(10),
    Ridge(alpha=1.0)
)
ridge_model.fit(X_train, y_train)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, title in zip(
    axes,
    [overfit_model, ridge_model],
    ["Degree 10 (No Regularization)", "Degree 10 + Ridge (alpha=1.0)"]
):
    y_pred_smooth = model.predict(X_true)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    
    ax.scatter(X_train, y_train, color="#3498db", s=40, label="Train")
    ax.scatter(X_test, y_test, color="#e74c3c", s=40, label="Test")
    ax.plot(X_true, y_true, "g--", linewidth=2, label="True function", alpha=0.7)
    ax.plot(X_true, y_pred_smooth, color="#8e44ad", linewidth=2, label="Prediction")
    ax.set_title(f"{title}\nTrain MSE: {train_mse:.3f} | Test MSE: {test_mse:.3f}")
    ax.set_ylim(-4, 12)
    ax.legend(fontsize=8)

plt.suptitle("Regularization: Taming Overfitting", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Tuning regularization strength
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
train_mses = []
val_mses = []

for alpha in alphas:
    model = make_pipeline(PolynomialFeatures(10), Ridge(alpha=alpha))
    
    model.fit(X_train, y_train)
    train_mses.append(mean_squared_error(y_train, model.predict(X_train)))
    
    val_scores = cross_val_score(model, X_train, y_train,
                                 cv=5, scoring="neg_mean_squared_error")
    val_mses.append(-val_scores.mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(alphas, train_mses, "o-", label="Train MSE", color="#3498db")
ax.semilogx(alphas, val_mses, "o-", label="Validation MSE", color="#e74c3c")
ax.set_xlabel("Regularization Strength (alpha)")
ax.set_ylabel("MSE")
ax.set_title("Effect of Regularization Strength")
ax.legend()
plt.tight_layout()
plt.show()

best_alpha = alphas[np.argmin(val_mses)]
print(f"Best alpha: {best_alpha}")
print(f"\nToo small alpha → overfitting")
print(f"Too large alpha → underfitting")
print(f"Sweet spot → best generalization")

## Summary: Bias-Variance Trade-off

| Scenario | Bias | Variance | Training Error | Test Error | Fix |
|----------|------|----------|----------------|------------|-----|
| Underfitting | High | Low | High | High | More complex model, more features |
| Overfitting | Low | High | Low | High | Regularization, more data, simpler model |
| Good fit | Low | Low | Low | Low | Goal achieved |

The key insight: **model complexity is a knob**. Too simple = underfitting. Too complex = overfitting. Use cross-validation to find the sweet spot.

## Key Takeaways

1. **Polynomial degree** controls model complexity — too high leads to overfitting.
2. **Learning curves** diagnose whether you have bias (underfitting) or variance (overfitting) problems.
3. **Cross-validation** gives robust performance estimates and helps select model complexity.
4. **Regularization** (Ridge/Lasso) controls overfitting by penalizing complex models.
5. **Alpha tuning** is essential — too much regularization causes underfitting.

Next: [exercise-01.md](../exercises/exercise-01.md) — practice these concepts.